# process_results_for_analysis.ipynb
- notebook purpose: read in raw output from sampler & preprocess into csvs with error rates for analysis
- python 3.13.7

In [1]:
import os, sys, regex as re
from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import statistics
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
os.chdir(os.path.expanduser("/Users/zeynepmarasli/Downloads/Documents/Projects/Daylong_Analyses/Python_Code/"))
sys.path.append("/Users/zeynepmarasli/Downloads/Documents/Projects/Daylong_Analyses/Python_Code/")
from module_lib.daylongtranscript import*

loaded


## Load transcripts

In [3]:
transcript_fpath = "/Users/zeynepmarasli/Downloads/Documents/Projects/Daylong_Analyses/Python_Code/Transcripts/For_OSF/"
A787_files = ["A787_001107_cleaned.txt", "A787_001109_cleaned.txt", "A787_001111_cleaned.txt"]

transcriptA1 = DaylongTranscript(fpath = transcript_fpath+A787_files[0], fname = A787_files[0], isVanDam=False)
transcriptA1.describe()
transcriptA2 = DaylongTranscript(fpath = transcript_fpath+A787_files[1], fname = A787_files[1], isVanDam=False)
transcriptA2.describe()
transcriptA3 = DaylongTranscript(fpath = transcript_fpath+A787_files[2], fname = A787_files[2], isVanDam=False)
transcriptA3.describe()

B895_files = ["B895_010002_cleaned.txt", "B895_010004_cleaned.txt"]
transcriptB1 = DaylongTranscript(fpath = transcript_fpath+B895_files[0], fname = B895_files[0], isVanDam=False)
transcriptB1.describe()

transcriptB2 = DaylongTranscript(fpath = transcript_fpath+B895_files[1], fname = B895_files[1], isVanDam=False)
transcriptB2.describe()

transcript_fpath = "/Users/zeynepmarasli/Downloads/Documents/Projects/Daylong_Analyses/Python_Code/Transcripts/For_OSF/"
fname = "BN32_clean.txt"
transcriptC = DaylongTranscript(fpath = transcript_fpath+fname, fname = fname, isVanDam = True)
transcriptC.describe()


Transcript:  A787_001107_cleaned.txt ---
	Audio Length:  45371456 ms //  756.191  min
	Total Word Count:  17411
	Intervals of silence: [[2895030, 37024008]]
			 34128978 ms // 568.816 min total silent intervals
			 187.375 min total speaking interval
Transcript:  A787_001109_cleaned.txt ---
	Audio Length:  40999367 ms //  683.323  min
	Total Word Count:  28123
	Intervals of silence: [[10515823, 15926622], [18771272, 33774998]]
			 20414525 ms // 340.242 min total silent intervals
			 343.081 min total speaking interval
Transcript:  A787_001111_cleaned.txt ---
	Audio Length:  42204520 ms //  703.409  min
	Total Word Count:  38531
	Intervals of silence: [[12545915, 14697605], [26935872, 34656022]]
			 9871840 ms // 164.531 min total silent intervals
			 538.878 min total speaking interval
Transcript:  B895_010002_cleaned.txt ---
	Audio Length:  44793268 ms //  746.554  min
	Total Word Count:  47192
	Intervals of silence: []
			 0 ms // 0.0 min total silent intervals
			 746.554 min total

In [4]:
TRANSCRIPTS = [transcriptA1, transcriptA2, transcriptA3, transcriptB1, transcriptB2, transcriptC]
TRANSCRIPT_LABELS_DICT = {transcriptA1: "A1", transcriptA2: "A2", transcriptA3: "A3", 
                    transcriptB1: "B1", transcriptB2: "B2", transcriptC: "C"}
TRANSCRIPTS_DICT = {"A787_001107_cleaned.txt": transcriptA1, "A787_001109_cleaned.txt": transcriptA2, "A787_001111_cleaned.txt": transcriptA3,
                    "B895_010002_cleaned.txt": transcriptB1, "B895_010004_cleaned.txt": transcriptB2, "BN32_clean.txt": transcriptC }

In [5]:
TRANSCRIPTA1_TRUEWC = transcriptA1.get_total_word_count()
TRANSCRIPTA2_TRUEWC = transcriptA2.get_total_word_count()
TRANSCRIPTA3_TRUEWC = transcriptA3.get_total_word_count()
TRANSCRIPTB1_TRUEWC = transcriptB1.get_total_word_count()
TRANSCRIPTB2_TRUEWC = transcriptB2.get_total_word_count()
TRANSCRIPTC_TRUEWC = transcriptC.get_total_word_count()
TRANSCRIPT_LABELS = ["A1", "A2", "A3", "B1", "B2", "C"]

In [ ]:
def grp_means_stds(group):
    #group = group.drop("Transcript", axis = 1)
    return group.mean(), group.std()

def get_feature_density(transcript, feature_dict):
    raw_amount = transcript.feature_count(feature_dict = feature_dict)
    denom_feature = 0
    # calculate density by either using overall word counts or utterance counts, depending on feature type 
    if list(feature_dict.keys())[0] == "utterance_annotation":
        denom_feature = len(transcript.utterances)
    else: denom_feature = transcript.get_total_word_count()
    density = raw_amount / denom_feature 
    return raw_amount, density 

def get_feature_densities(feature_dicts, transcripts = TRANSCRIPTS, transcript_labels = TRANSCRIPT_LABELS):
    '''
    - wrapper function for get_feature_density'''
    C_xds_codes = {"T": "id", "A": "od", "C": None, 'O': None, 'U': None}
    densities = []
    for feature_dict in feature_dicts:
        feature_type = list(feature_dict.keys())[0]
        feature = list(feature_dict.values())[0]  
        indiv_feature_density = []     
        print(f"Feature densities for: {feature_type}, {feature}")
        for i, transcript in enumerate(transcripts):
            if transcript_labels[i] == "C": 
                if feature_type == "utterance_annotation": continue 
                elif feature_type == "xds": 
                    feature = C_xds_codes[feature]
                    feature_dict = {"xds": feature}
            raw_amount, density = get_feature_density(transcript, feature_dict)
            indiv_feature_density.append(density)
            densities.append(density)
            print(f"\t{transcript_labels[i]:<5}: {raw_amount:<5} \t {density * 100}%")
        if len(indiv_feature_density) > 1: print(f"\tOverall: {statistics.mean(indiv_feature_density) * 100:<5}% \tStdev: {statistics.stdev(indiv_feature_density)*100}%")
    #print(f"\nGroup Mean: {statistics.mean(densities)*100:<5}% \tStdev: {statistics.stdev(densities)*100}%")
    return(densities)

# Part 1: Simulation & Evaluation of Sampling Interval Size & Total Sampled Time
- implementation of random sampling with replacement procedure

In [16]:
TOTAL_TS = ["30", "40", "50", "60", "70", "80", "90", "100", "110", "120", "130", '140', '150', '160', '170', '180'] # in minutes 
PROP_TTS = ["0.05","0.1", "0.15", "0.2", "0.3", "0.4", "0.5", "0.6", "0.7", "0.8", "0.9"] 
TOTAL_TS_3HR = ["130", '140', '150', '160', '170', '180']

In [17]:
def prop_of_samples(df, cols):
    ''' 
    - given a dataframe & total time sampled, will output proportion of samples greater than 2%, 5%, 10%, and 20% error
    '''
    print("Proportion of samples > x% error")
    for col in cols :
        temp_df = df[col]
        samples_2 = len(temp_df[temp_df>2]) / 600
        samples_5 = len(temp_df[temp_df>5]) / 600
        samples_10 = len(temp_df[temp_df>10]) / 600
        samples_20 = len(temp_df[temp_df>20]) / 600
        print(f'--{col} group\n> 2%: {samples_2:<25}\t> 5%: {samples_5:<25}\t > 10%: {samples_10:<25}\t>20%: {samples_20}')
        print()
    return

### Analysis 1.1: Sampling Interval Size: 30 seconds to 60 minutes 

In [18]:
SIM_TYPE = ["30 seconds", "1 minute", "5 minutes", "10 minutes", "30 minutes", "60 minutes"]

In [19]:
#returns correct variable name given interval length
def get_simulation_type(interval_length):
    if interval_length == 30000.0: return 0
    if interval_length == 60000.0: return 1
    if interval_length == 300000.0: return 2
    if interval_length == 600000.0: return 3
    if interval_length == 1800000.0: return 4
    if interval_length == 3600000.0: return 5 

# process data into a list of lists 
def process_data(data):
    sim_30000 = []   # 30 seconds
    sim_60000 = []  # 1 minute
    sim_300000 = []  # 5 minutes
    sim_600000 = []   # 10 minutes
    sim_1800000 = [] # 30 minutes --> will only have 4 data points (30 min, 60 min, 90 min, 120 min)
    sim_3600000 = []  # 60 minutes --> will only have 2 data points (60 min, 120 min)
    all_simulations = [sim_30000, sim_60000, sim_300000, sim_600000, sim_1800000, sim_3600000]
    for sim_type in all_simulations: sim_type = [] # resest all lists to empty 
    sim_type = None 
    estimates = []
    transcript_filename = None
    for index, line in enumerate(data):
        #print(line)
        tokens = line.split() 
        if len(tokens) == 0: pass
        elif tokens[0] == "Transcript:": 
            transcript_filename = tokens[1]
            continue #go to next line 
        elif len(tokens) == 2:
            sim_index = get_simulation_type(float(tokens[0]))
            sim_type = all_simulations[sim_index]
        else: #have line with estimates
            estimates = [float(i) for i in tokens] 
            sim_type.append(estimates)
    
    # sim 30 min & 60 min do not have all data points: rearrange data points so that:
        # sim 30 min = [esimate, Na, Na, estimate, Na, Na, estimate, Na, Na, estimate]
        # sim 60 min = [Na, Na, Na, estimate, Na, Na, Na, Na, Na, estimate]
    none_list = [None] * len(sim_30000)
    temp = [none_list] * 10
    temp[0] = sim_1800000[0]
    temp[3] = sim_1800000[1]
    temp[6] = sim_1800000[2]
    temp[9] = sim_1800000[3]
    all_simulations[4] = temp 
    
    none_list = [None] * len(sim_30000)
    temp = [none_list] * 10
    temp[3] = sim_3600000[0]
    temp[9] = sim_3600000[1]
    all_simulations[5] = temp 

    # get transcript groundtruth & label
    transcript = TRANSCRIPTS_DICT[transcript_filename]
    groundtruth = transcript.get_total_word_count()
    transcript_label = TRANSCRIPT_LABELS_DICT[transcript]
    return all_simulations, groundtruth, transcript_label

In [20]:
def make_dataframes(data):
    all_df_raw = []
    all_means = []
    all_stdevs = []
    for simtype in data:
        df = simtype_dataframe(simtype)
        #means = get_means(df)
        #stdevs = get_sd(df)
        all_df_raw.append(df)
        #all_means.append(means)
        #all_stdevs.append(stdevs)
    #all_df_means = stats_df(all_means)
    #all_df_sd = stats_df(all_stdevs)
    #all_df_raw["Transcript"]
    return all_df_raw#, all_df_means, all_df_sd

# input: raw datapoints from simulation type (interval length)
# returns: dataframe; columns: total time sampled 
                    # rows:    data points (ex. daylong estimate)
def simtype_dataframe(simtype):
    ts = ["30", "40", "50", "60", "70", "80", "90", "100", "110", "120"]
    df = pd.DataFrame(simtype)
    df = df.T 
    df.columns = ts  
    return df 

def convert_to_min(value):
    return (value / (60 * 1000))

def get_means(raw_df):
    means = []
    for col in raw_df.columns:
        means.append(statistics.mean(raw_df[col]))
    return means 

def get_sd(raw_df):
    stdevs = []
    for col in raw_df.columns:
        stdevs.append(statistics.pstdev(raw_df[col]))
    return stdevs 

def stats_df(list):
    df = pd.DataFrame(list)
    df = df.T
    df.columns = SIM_TYPE
    df.index = TOTAL_TS
    return df  

def get_percent_error(data_raw, true_WC):
    all_df_perr = []
    #perr_means = []
    #perr_std = [] 
    for df in data_raw:
        df_err = abs( (df - true_WC ) / true_WC )
        #df_err = (df - true_WC ) / true_WC 
        df_perr = round((df_err * 100), 3) 
        all_df_perr.append(df_perr)
        #perr_means.append(df_perr.mean())
        #perr_std.append(df_perr.std())
        
    """df_perr_means = (pd.DataFrame(perr_means)).T
    df_perr_means.columns = SIM_TYPE
    
    df_perr_std = (pd.DataFrame(perr_std)).T
    df_perr_std.columns = SIM_TYPE"""
    
    return all_df_perr#, df_perr_means, df_perr_std]

def make_percent_error_data_1(filenamepath):
    file = open(filenamepath, 'r')
    data = file.readlines()
    all_simulations, groundtruth, transcript_label = process_data(data)
    all_df_raw =  make_dataframes(all_simulations) #, all_df_means, all_df_sd
    perr_all_dfs =  get_percent_error(all_df_raw, groundtruth)

    combined_dfs = []
    for i, sampling_int in enumerate(SIM_TYPE): 
        perr_samplingint = perr_all_dfs[i]
        perr_samplingint["Sampling Interval Size"] = sampling_int
        combined_dfs.append(perr_samplingint)
    perr_samplingint_df = pd.concat(combined_dfs, ignore_index=True)
    perr_samplingint_df["Transcript"] = transcript_label
    return perr_samplingint_df

In [21]:
def reformat_sampintsize_results(df, tts = "120"): 
    df = df[[tts, "Transcript", "Sampling Interval Size"]]
    # create a helper index for duplicates within each group
    df["dup_id"] = df.groupby(["Transcript", "Sampling Interval Size"]).cumcount()

    # pivot using the helper index
    df_wide = df.pivot(
        index=["Transcript", "dup_id"],
        columns="Sampling Interval Size",
        values=tts
    ).reset_index()

    # optional: drop helper column if you don't want it
    df_wide = df_wide.drop(columns="dup_id")
    df_wide.columns.name = None
    df_wide = df_wide.reindex(columns=["30 seconds", "1 minute", "5 minutes", "10 minutes", "30 minutes", "60 minutes"] + ["Transcript"])
    df_wide
    return df_wide

In [22]:
perr_dfs =[]
source_dir = Path("DaylongSampling/Results/RandomSamplingwithReplacement/SamplingIntervalSize/")
files = source_dir.glob('*_TEST_daylong_estimates.txt')
for file in files: 
    print(file)
    df = make_percent_error_data_1(file)
    perr_dfs.append(df)
perr_allsamplingint = pd.concat(perr_dfs, ignore_index=True)
perr_samplingintsize = reformat_sampintsize_results(perr_allsamplingint)

DaylongSampling/Results/RandomSamplingwithReplacement/SamplingIntervalSize/B895_010004_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/SamplingIntervalSize/A787_001107_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/SamplingIntervalSize/B895_010002_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/SamplingIntervalSize/BN32_clean.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/SamplingIntervalSize/A787_001111_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/SamplingIntervalSize/A787_001109_cleaned.txt_word count_TEST_daylong_estimates.txt


/var/folders/n_/nwpv122n5g586b70zt2jx3vw0000gn/T/ipykernel_83115/1452690401.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["dup_id"] = df.groupby(["Transcript", "Sampling Interval Size"]).cumcount()


In [ ]:
# write to csv file
# perr_samplingintsize.to_csv(path_or_buf='DaylongSampling/Results/CSV/Analysis1_1_SamplingIntervalSize_EstimatePercentError.csv', sep = ",", header = True, index = False)

### Analysis 1.2 Total Time Sampled 

In [35]:
def convert_to_min(value):
    return (value / (60 * 1000))

def get_tts_keys(prop_tts = True):
    if prop_tts is True: return PROP_TTS
    else: return TOTAL_TS

def get_groundtruth(transcript, feature = None, feature_arg = None):
    if feature_arg == "word count": return transcript.get_total_word_count()
    if feature == "count": return transcript.get_total_word_count()
    else:
        # select feature  
        feature_spec = re.split(r'[()]', feature)
        feature_type = feature_spec[0]
        feature = feature_spec[1]
        return transcript.feature_count(feature_dict = {feature_type:feature})

def process_raw_data(filenamepath, prop_tts = True, feature_arg = None):
    """
    - given lines from raw file, read in line by line and extract appropriate data
    - expected format: 
        \nTranscript: [transcript name] Sampling_interval: [value] Prop_TTS: [value] TTS:[value] Feature: [value]
        \n[sampling interval size] [# of samples]
        \n[estimate value]
    - process as proportion total sampled time
    - returns: 1. DataFrame of <tts key> <values> 2. groundtruth value 3. transcript label
    - 
    """
    file = open(filenamepath, 'r')
    lines = file.readlines()

    if prop_tts: 
        #total_time_sampled = PROP_TTS
        tts_key_index = 5
    else: 
        #total_time_sampled = TOTAL_TS
        tts_key_index = 7

    #keys = [float(key) for key in total_time_sampled]
    transcript_filename = None
    feature = None
    dict = {}
    curr_key = ""
    for line in lines:       
        tokens = line.split()
        if tokens[0] == "Transcript:": ## new TTS key 
            curr_key = float(tokens[tts_key_index])
            if prop_tts == False: curr_key = int(convert_to_min(curr_key))
            transcript_filename = tokens[1]
            feature = tokens[-1]
            continue
        elif len(tokens) == 0: 
            continue
        elif len(tokens) == 2: 
            continue # don't need this line
        else: 
            tokens = [float(token) for token in tokens]
            dict.setdefault(curr_key, []).extend(tokens)

    # get transcript groundtruth & label
    transcript = TRANSCRIPTS_DICT[transcript_filename]
    groundtruth = get_groundtruth(transcript, feature, feature_arg)
    transcript_label = TRANSCRIPT_LABELS_DICT[transcript]
    return dict, groundtruth, transcript_label

def calc_percent_error(estimate, groundtruth):
    error = abs( ( estimate - groundtruth ) / groundtruth )
    return round( ( error * 100), 3)

def get_percent_errors(raw_data, groundtruth, prop_tts= True):
    total_time_sampled = get_tts_keys(prop_tts)
    keys = [float(key) for key in total_time_sampled]
    dict = {}
    #dict = {key: [] for key in keys}

    for key, values in raw_data.items():
        percent_errors = map(calc_percent_error, values,itertools.repeat(groundtruth, len(values)) )
        dict.setdefault(key, []).extend(percent_errors)
    return dict

def get_percenterror_df(filepath, prop_tts = True, feature_arg = None):
    """
    - wrapper method to get dataframes for means & stds of percent estimate errors 
    - input: filepath to raw data file & ground truth value for feature
    - returns: 1 dataframes  
    """
    raw_data, groundtruth, transcript_label = process_raw_data(filepath, prop_tts=prop_tts, feature_arg = feature_arg)
    percent_errors_all = get_percent_errors(raw_data, groundtruth, prop_tts = prop_tts)
    percent_errors_all = pd.DataFrame(percent_errors_all)
    percent_errors_all["Transcript"] = transcript_label
    return  percent_errors_all

#### as total minutes sampled 
    - 30 to 180 minutes sampled
    - sampling int size: 30 seconds

In [44]:
# get 30 to 120 min tts data 
TOTAL_TS = ["30", "40", "50", "60", "70", "80", "90", "100", "110", "120", "130", '140', '150', '160', '170', '180']
perr_dfs =[]
source_dir = Path("DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_RawMinutes/TTS_120Min/")
files = source_dir.glob('*_TEST_daylong_estimates.txt')
for file in files: 
    print(file)
    df = make_percent_error_data_1(file)
    perr_dfs.append(df)
perr_allsamplingint_30to120min = pd.concat(perr_dfs, ignore_index=True)

DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_RawMinutes/TTS_120Min/B895_010004_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_RawMinutes/TTS_120Min/A787_001107_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_RawMinutes/TTS_120Min/B895_010002_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_RawMinutes/TTS_120Min/BN32_clean.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_RawMinutes/TTS_120Min/A787_001111_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_RawMinutes/TTS_120Min/A787_001109_cleaned.txt_word count_TEST_daylong_estimates.txt


In [45]:
# get 120 to 180 min tts data 
TOTAL_TS = TOTAL_TS_3HR
perr_dfs =[]
source_dir = Path("DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_RawMinutes/TTS_130Min/")
files = source_dir.glob('*_TEST_daylong_estimates.txt')
for file in files: 
    print(file)
    df = get_percenterror_df(file, prop_tts = False, feature_arg = "word count")
    perr_dfs.append(df)
combined_perr_mintts_df_3hr = pd.concat(perr_dfs, ignore_index=True)


DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_RawMinutes/TTS_130Min/B895_010004_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_RawMinutes/TTS_130Min/A787_001107_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_RawMinutes/TTS_130Min/B895_010002_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_RawMinutes/TTS_130Min/BN32_clean.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_RawMinutes/TTS_130Min/A787_001111_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_RawMinutes/TTS_130Min/A787_001109_cleaned.txt_word count_TEST_daylong_estimates.txt


In [46]:
# combine with 30-180 min
TOTAL_TS = ["30", "40", "50", "60", "70", "80", "90", "100", "110", "120", "130", '140', '150', '160', '170', '180']
combined_perr_mintts_df_120min = (perr_allsamplingint_30to120min[perr_allsamplingint_30to120min["Sampling Interval Size"] == "30 seconds"]).drop(columns = ["Sampling Interval Size", "Transcript"]).reset_index(drop=True)
combined_perr_mintts_df = pd.concat([combined_perr_mintts_df_120min, combined_perr_mintts_df_3hr], axis=1)
combined_perr_mintts_df.columns = [30,40,50,60,70,80,90,100,110,120,130,140,150,160,170,180] + ["Transcript"]
combined_perr_mintts_df.describe()

,30,40,50,60,70,80,90,100,110,120,130,140,150,160,170,180
count,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000
mean,8.534370,7.390060,6.280665,5.885092,5.666157,5.449280,5.088098,4.784178,4.626010,4.289675,4.141982,4.287602,4.133303,3.794250,3.710998,3.625385
std,7.220213,6.374735,5.054117,4.922857,4.371144,4.454926,3.967874,3.779529,3.797086,3.498239,3.319959,3.427313,3.374915,3.253985,2.943305,2.911623
min,0.020000,0.005000,0.011000,0.002000,0.034000,0.014000,0.006000,0.029000,0.007000,0.006000,0.015000,0.003000,0.015000,0.012000,0.041000,0.002000
25%,3.384750,2.637500,2.544000,2.044500,2.234000,2.023000,2.020000,1.824000,1.692000,1.566250,1.637500,1.713750,1.535000,1.431250,1.462000,1.374000
50%,6.693000,5.631000,5.091500,4.674500,4.771000,4.315500,4.378500,3.864000,3.574000,3.594000,3.399500,3.488000,3.323000,2.966500,3.099000,2.940500
75%,12.434750,10.182250,8.701750,8.582750,7.943750,7.690000,6.959500,7.047500,6.657750,6.033750,5.706500,5.891250,6.042500,5.238250,5.132750,5.248000
max,55.151000,34.507000,28.597000,29.411000,23.573000,22.522000,24.189000,22.475000,19.447000,22.707000,18.634000,19.987000,21.283000,20.279000,15.503000,16.001000


In [ ]:
# write to csv file
# combined_perr_mintts_df.to_csv(path_or_buf='DaylongSampling/Results/CSV/Analysis1_2_TotalTimeSampled_asMinutes_EstimatePercentError.csv', sep = ",", header = True, index = False)

#### as proportion of total recording time
 - 5% to 90% of recording sampled

In [49]:
perr_dfs =[]
source_dir = Path("DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_ProportionbyTotalAudioTime/")
files = source_dir.glob('*_TEST_daylong_estimates.txt')
for file in files: 
    print(file)
    df = get_percenterror_df(file, prop_tts = True, feature_arg="word count")
    perr_dfs.append(df)

DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_ProportionbyTotalAudioTime/B895_010004_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_ProportionbyTotalAudioTime/A787_001107_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_ProportionbyTotalAudioTime/B895_010002_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_ProportionbyTotalAudioTime/BN32_clean.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_ProportionbyTotalAudioTime/A787_001111_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/TotalTimeSampled_ProportionbyTotalAudioTime/A787_001109_cleaned.txt_word count_TEST_daylong_estimates.txt


In [50]:
combined_perr_proptts_df = pd.concat(perr_dfs, ignore_index=True)
combined_perr_proptts_df = combined_perr_proptts_df.reindex(columns = [0.05, 0.1, 0.12, 0.15, 0.18, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, "Transcript"]).drop(columns = [0.12, 0.18])
combined_perr_proptts_df

,0.05,0.1,0.15,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,Transcript
0,13.271,6.343,6.323,1.051,5.343,10.084,2.147,0.973,0.005,2.413,2.583,B2
1,11.734,4.594,6.696,4.911,2.526,0.170,0.584,0.959,4.231,1.302,3.440,B2
2,14.718,17.332,12.123,6.693,3.754,1.564,2.565,7.401,0.055,1.990,1.932,B2
3,15.930,8.958,2.621,9.364,3.010,3.463,3.234,0.437,2.227,0.829,0.649,B2
4,13.227,1.261,3.910,0.736,0.421,5.133,0.076,1.748,3.107,1.938,0.683,B2
...,...,...,...,...,...,...,...,...,...,...,...,...
595,0.845,0.155,0.226,0.988,0.286,1.220,1.901,0.232,0.544,2.217,1.027,A2
596,4.868,0.649,3.237,0.289,2.666,0.780,0.065,0.637,0.147,1.097,0.484,A2
597,6.272,4.291,1.940,1.056,3.243,0.418,1.876,2.244,1.323,1.311,0.799,A2
598,3.083,1.387,0.071,6.680,1.565,1.729,1.227,1.574,0.983,3.188,0.484,A2


In [ ]:
# write to csv file
# combined_perr_proptts_df.to_csv(path_or_buf='DaylongSampling/Results/CSV/Analysis1_2_TotalTimeSampled_asProp_EstimatePercentError.csv', sep = ",", header = True, index = False)

### Analysis 1.3: Select Features

In [55]:
def make_perrdata_featuredensitygrp(sourcedirs, select_features, density_group, 
                                    source_dir = "DaylongSampling/Results/RandomSamplingwithReplacement/Select_Features/"):
    densitygrp_combined_df = []
    print(density_group, " ", select_features)
    for i, select_feature in enumerate(select_features):
        print(select_feature)
        source_dir_full = Path(f"{source_dir}{sourcedirs[i]}/")
        files = source_dir_full.glob('*_TEST_daylong_estimates.txt')
        perr_dfs = []
        for file in files: 
            print(file)
            df = get_percenterror_df(file, prop_tts = True)
            perr_dfs.append(df)
        combined_df = pd.concat(perr_dfs, ignore_index=True)
        combined_df["Feature"] = select_feature 
        densitygrp_combined_df.append(combined_df)
    densitygrp_combined_df = pd.concat(densitygrp_combined_df,  ignore_index = True)
    densitygrp_combined_df["Frequency Density Group"] = density_group
    return densitygrp_combined_df


In [56]:
PROP_TTS = ["0.05","0.1", "0.15","0.2", "0.3", "0.4", "0.5", "0.6", "0.7", "0.8", "0.9"] 
sourcedirs = [["Rare_Density/Select_Word", "Rare_Density/Utterance_Imitations"], 
              ["Low_Density/Utterance_Exclamations", "Low_Density/Speaker_CHI", "Low_Density/XDS_C"],
              ["Medium_Density/Utterance_Questions", "Medium_Density/Speaker_FC1", "Medium_Density/XDS_T"], 
              ["High_Density/Speaker_FA1_MA1", "High_Density/Speaker_EE1", "High_Density/XDS_A_V2", "High_Density/XDS_T"],
              ["Very_High_Density/Speaker_FA1", "Very_High_Density/XDS_A", "Very_High_Density/XDS_C"]]
select_features = [["select word", "imitations"], 
                   ["exclamations", "target-child vocalizations", "other-child directed speech"],
                   ["questions", "speech from non-target child", "target-child directed speech"], 
                   ["speech from FA1/MA1", "electronic source", "adult directed speech", "target-child directed speech"],
                   ["speech from FA1", "adult directed speech", "other-child directed speech"]]
freqdensity = ["rare", "low", "medium", "high", "very high"]
perr_freqdensity_groups = pd.DataFrame()
for i, fd_group in enumerate(freqdensity):
    fd_group_perr = make_perrdata_featuredensitygrp(sourcedirs = sourcedirs[i], select_features = select_features[i], density_group = fd_group)
    perr_freqdensity_groups = pd.concat([perr_freqdensity_groups, fd_group_perr],  ignore_index = True)
perr_freqdensity_groups = perr_freqdensity_groups.reindex(columns = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, "Transcript", "Feature", "Frequency Density Group"])

rare   ['select word', 'imitations']
select word
DaylongSampling/Results/RandomSamplingwithReplacement/Select_Features/Rare_Density/Select_Word/B895_010002_cleaned.txt_select_word_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/Select_Features/Rare_Density/Select_Word/A787_001111_cleaned.txt_select_word_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/Select_Features/Rare_Density/Select_Word/A787_001109_cleaned.txt_select_word_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/Select_Features/Rare_Density/Select_Word/A787_001107_cleaned.txt_select_word_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/Select_Features/Rare_Density/Select_Word/B895_010004_cleaned.txt_select_word_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithReplacement/Select_Features/Rare_Density/Select_Word/BN32_clean.txt_select_word_TEST_daylong_estimates.txt
imitations
D

In [57]:
perr_freqdensity_groups

,0.05,0.1,0.15,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,Transcript,Feature,Frequency Density Group
0,50.000,25.000,66.675,12.500,75.000,37.500,50.000,33.325,14.275,25.000,22.225,B1,select word,rare
1,100.000,25.000,66.675,37.500,33.325,18.750,25.000,33.325,3.575,6.250,8.325,B1,select word,rare
2,350.000,75.000,50.000,62.500,25.000,12.500,5.000,33.325,35.725,9.375,2.775,B1,select word,rare
3,100.000,25.000,16.675,37.500,116.675,43.750,35.000,16.675,25.000,31.250,25.000,B1,select word,rare
4,250.000,0.000,50.000,37.500,58.325,18.750,45.000,16.675,10.725,0.000,16.675,B1,select word,rare
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5495,10.293,9.942,10.555,4.285,6.237,0.548,4.517,1.311,2.170,2.364,1.670,A2,other-child directed speech,very high
5496,1.292,9.707,11.859,5.500,4.605,2.788,2.786,1.154,1.264,3.473,3.766,A2,other-child directed speech,very high
5497,9.277,5.950,4.554,2.310,1.983,2.085,2.787,5.759,0.079,0.113,1.234,A2,other-child directed speech,very high
5498,9.433,12.016,6.040,2.094,3.091,4.286,2.435,0.143,5.826,2.877,3.771,A2,other-child directed speech,very high


In [ ]:
# write to csv
# perr_freqdensity_groups.to_csv(path_or_buf='DaylongSampling/Results/CSV/Analysis1_3_SelectFeatures_EstimatePercentError.csv', sep = ",", header = True, index = False)

# Part 2: Evaluation and Simulation of Sampling Procedures
- Systematic Sampling, Random w/out Replacement, Random w/ Replacement

In [59]:
def clean_for_systematicsampling(raw_df):
    df = raw_df.copy()
    prop_tts = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4]
    nan_columns = {'A1': 0.2, 'A2': 0.3, 'A3':0.4, 'B1': 0.4, 'B2': 0.4, 'C': 0.4,}
    df = df[df.columns.intersection(prop_tts + ["Transcript"])]
    for key_transcript, value_col in nan_columns.items():
        idx = df.columns.get_loc(value_col)
        if prop_tts[idx] == prop_tts[-1] : pass
        else: df.loc[df["Transcript"] == key_transcript, prop_tts[idx+1:]] = np.nan
    return df


## Systematic Sampling

### Estimating: Daylong Word Counts

In [61]:
# read in 30 second sampling interval data
perr_dfs =[]
source_dir = Path("DaylongSampling/Results/SystematicSampling/Prop_TTS/30SecSamplingInterval/")
files = source_dir.glob('*_TEST_daylong_estimates.txt')
for file in files: 
    print(file)
    df = get_percenterror_df(file, prop_tts = True, feature_arg="word count")
    perr_dfs.append(df)
combined_perr_proptts_systematic_30s = pd.concat(perr_dfs, ignore_index=True)
combined_perr_proptts_systematic_30s = combined_perr_proptts_systematic_30s.reindex(columns = [0.05, 0.1, 0.12, 0.15, 0.18, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, "Transcript"])
combined_perr_proptts_systematic_30s = clean_for_systematicsampling(combined_perr_proptts_systematic_30s)
combined_perr_proptts_systematic_30s["Sampling Interval"] = "30 seconds"
combined_perr_proptts_systematic_30s.describe()

DaylongSampling/Results/SystematicSampling/Prop_TTS/30SecSamplingInterval/B895_010004_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/SystematicSampling/Prop_TTS/30SecSamplingInterval/A787_001107_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/SystematicSampling/Prop_TTS/30SecSamplingInterval/B895_010002_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/SystematicSampling/Prop_TTS/30SecSamplingInterval/BN32_clean.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/SystematicSampling/Prop_TTS/30SecSamplingInterval/A787_001111_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/SystematicSampling/Prop_TTS/30SecSamplingInterval/A787_001109_cleaned.txt_word count_TEST_daylong_estimates.txt


,0.05,0.1,0.15,0.2,0.3,0.4
count,600.000000,600.000000,600.000000,600.000000,500.000000,400.000000
mean,5.161447,4.042315,2.446522,1.939030,1.507294,1.217675
std,4.027497,2.305025,1.566184,1.328896,0.915749,0.789023
min,0.070000,0.180000,0.027000,0.254000,0.032000,0.406000
25%,1.815000,2.221000,1.628000,0.850000,0.762000,0.442000
50%,4.663000,3.772000,2.239000,1.596000,1.671000,1.142000
75%,7.157000,5.864000,2.900000,2.239000,2.280000,1.595000
max,20.488000,9.280000,6.183000,5.286000,2.761000,3.234000


In [62]:
# read in 5 minute sampling interval data
perr_dfs =[]
source_dir = Path("DaylongSampling/Results/SystematicSampling/Prop_TTS/5MinSamplingInterval/")
files = source_dir.glob('*_TEST_daylong_estimates.txt')
for file in files: 
    print(file)
    df = get_percenterror_df(file, prop_tts = True, feature_arg="word count")
    perr_dfs.append(df)
combined_perr_proptts_systematic_5min = pd.concat(perr_dfs, ignore_index=True)
combined_perr_proptts_systematic_5min = combined_perr_proptts_systematic_5min.reindex(columns = [0.05, 0.1, 0.12, 0.15, 0.18, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, "Transcript"])
combined_perr_proptts_systematic_5min = clean_for_systematicsampling(combined_perr_proptts_systematic_5min)
combined_perr_proptts_systematic_5min["Sampling Interval"] = "5 minutes"
combined_perr_proptts_systematic_5min.head()

DaylongSampling/Results/SystematicSampling/Prop_TTS/5MinSamplingInterval/B895_010004_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/SystematicSampling/Prop_TTS/5MinSamplingInterval/A787_001107_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/SystematicSampling/Prop_TTS/5MinSamplingInterval/B895_010002_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/SystematicSampling/Prop_TTS/5MinSamplingInterval/BN32_clean.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/SystematicSampling/Prop_TTS/5MinSamplingInterval/A787_001111_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/SystematicSampling/Prop_TTS/5MinSamplingInterval/A787_001109_cleaned.txt_word count_TEST_daylong_estimates.txt


,0.05,0.1,0.15,0.2,0.3,0.4,Transcript,Sampling Interval
0,11.609,27.204,10.645,0.862,1.371,1.670,B2,5 minutes
1,11.609,4.346,10.645,20.583,0.250,1.670,B2,5 minutes
2,13.745,9.366,10.645,7.350,1.371,3.244,B2,5 minutes
3,11.609,11.089,17.200,7.350,0.250,3.244,B2,5 minutes
4,11.609,9.366,17.880,0.862,0.741,1.670,B2,5 minutes


In [63]:
combined_perr_proptts_systematic = pd.concat([combined_perr_proptts_systematic_30s, combined_perr_proptts_systematic_5min],ignore_index=True)
combined_perr_proptts_systematic

,0.05,0.1,0.15,0.2,0.3,0.4,Transcript,Sampling Interval
0,16.213,5.909,0.343,0.581,0.762,1.595,B2,30 seconds
1,8.152,3.384,0.343,3.771,0.101,1.142,B2,30 seconds
2,3.395,6.203,3.965,1.568,1.542,1.142,B2,30 seconds
3,20.488,1.503,3.965,3.853,0.101,1.142,B2,30 seconds
4,3.414,5.909,2.661,1.568,0.762,1.595,B2,30 seconds
...,...,...,...,...,...,...,...,...
1195,37.540,4.803,2.446,2.936,0.841,NaN,A2,5 minutes
1196,11.136,11.596,1.903,2.506,0.841,NaN,A2,5 minutes
1197,18.049,11.596,2.446,4.135,2.183,NaN,A2,5 minutes
1198,3.788,4.803,9.697,2.936,0.841,NaN,A2,5 minutes


In [ ]:
# write to csv
# combined_perr_proptts_systematic.to_csv(path_or_buf='DaylongSampling/Results/CSV/Analysis2_1_SystematicSampling_EstimatePercentError.csv', sep = ",", header = True, index = False)

### Estimating: Select Features

In [64]:
freqdensity = ["rare", "low", "medium", "high", "very high"]
PROP_TTS = ["0.05","0.1", "0.2", "0.3", "0.4", "0.5", "0.6", "0.7", "0.8", "0.9"] 
source_dir = "DaylongSampling/Results/SystematicSampling/Prop_TTS/Select_Features/"
sourcedirs = [["Rare_Density/Select_Word", "Rare_Density/Utterance_Imitations"], 
              ["Low_Density/Utterance_Exclamations", "Low_Density/Speaker_CHI", "Low_Density/XDS_C"],
              ["Medium_Density/Utterance_Questions", "Medium_Density/Speaker_FC1", "Medium_Density/XDS_T"], 
              ["High_Density/Speaker_FA1_MA1", "High_Density/Speaker_EE1", "High_Density/XDS_A", "High_Density/XDS_T"],
              ["Very_High_Density/Speaker_FA1", "Very_High_Density/XDS_A", "Very_High_Density/XDS_C"]]
select_features = [["select word", "imitations"], 
                   ["exclamations", "target-child vocalizations", "other-child directed speech"],
                   ["questions", "speech from non-target child", "target-child directed speech"], 
                   ["speech from FA1/MA1", "electronic source", "adult directed speech", "target-child directed speech"],
                   ["speech from FA1", "adult directed speech", "other-child directed speech"]]
freqdensity = ["rare", "low", "medium", "high", "very high"]
perr_freqdensity_groups_systematic = pd.DataFrame()
for i, fd_group in enumerate(freqdensity):
    fd_group_perr = make_perrdata_featuredensitygrp(source_dir = source_dir,sourcedirs = sourcedirs[i], select_features = select_features[i], density_group = fd_group)
    perr_freqdensity_groups_systematic = pd.concat([perr_freqdensity_groups_systematic, fd_group_perr],  ignore_index = True)
#perr_freqdensity_groups_systematic = clean_systematicsampling_proptts(perr_freqdensity_groups_systematic, prop_tts = list(perr_freqdensity_groups_systematic.columns)[0:-1])
#perr_freqdensity_groups_systematic = perr_freqdensity_groups_systematic.reindex(columns = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, "Transcript", "Feature", "Frequency Density Group"])


rare   ['select word', 'imitations']
select word
DaylongSampling/Results/SystematicSampling/Prop_TTS/Select_Features/Rare_Density/Select_Word/B895_010002_cleaned.txt_select_word_TEST_daylong_estimates.txt
DaylongSampling/Results/SystematicSampling/Prop_TTS/Select_Features/Rare_Density/Select_Word/A787_001111_cleaned.txt_select_word_TEST_daylong_estimates.txt
DaylongSampling/Results/SystematicSampling/Prop_TTS/Select_Features/Rare_Density/Select_Word/A787_001109_cleaned.txt_select_word_TEST_daylong_estimates.txt
DaylongSampling/Results/SystematicSampling/Prop_TTS/Select_Features/Rare_Density/Select_Word/A787_001107_cleaned.txt_select_word_TEST_daylong_estimates.txt
DaylongSampling/Results/SystematicSampling/Prop_TTS/Select_Features/Rare_Density/Select_Word/B895_010004_cleaned.txt_select_word_TEST_daylong_estimates.txt
DaylongSampling/Results/SystematicSampling/Prop_TTS/Select_Features/Rare_Density/Select_Word/BN32_clean.txt_select_word_TEST_daylong_estimates.txt
imitations
DaylongSampli

In [65]:
df_values = perr_freqdensity_groups_systematic.iloc[:, 0:12]
df_labels = perr_freqdensity_groups_systematic.iloc[:, 11:]
df_values_clean = clean_for_systematicsampling(df_values)
perr_freqdensity_groups_systematic_cleaned = pd.concat([df_values_clean.iloc[:, 0:-1], df_labels], axis=1)
perr_freqdensity_groups_systematic_cleaned.head()

,0.05,0.1,0.15,0.2,0.3,0.4,Transcript,Feature,Frequency Density Group
0,100.000,72.550,92.775,19.925,0.075,4.925,B1,select word,rare
1,47.425,37.225,65.125,40.025,9.775,19.925,B1,select word,rare
2,5.150,9.775,12.375,4.925,19.950,4.925,B1,select word,rare
3,5.150,174.475,92.775,4.925,19.950,25.050,B1,select word,rare
4,57.725,17.650,65.125,4.925,19.950,25.050,B1,select word,rare


In [ ]:
# write to csv
# perr_freqdensity_groups_systematic_cleaned.to_csv(path_or_buf='DaylongSampling/Results/CSV/Analysis2_2_SystematicSampling_SelectFeatures_EstimatePercentError.csv', sep = ",", header = True, index = False)

## Random Sampling w/ Replacement (comparison)

### Estimating: Daylong Word Counts

In [318]:
# for 30 seconds, just need to clean combined_perr_proptts_df
combined_perr_proptts_random_30s = clean_for_systematicsampling(combined_perr_proptts_df)
combined_perr_proptts_random_30s["Sampling Interval"] = "30 seconds"
combined_perr_proptts_random_30s.head()

,0.05,0.1,0.15,0.2,0.3,0.4,Transcript,Sampling Interval
0,13.271,6.343,6.323,1.051,5.343,10.084,B2,30 seconds
1,11.734,4.594,6.696,4.911,2.526,0.170,B2,30 seconds
2,14.718,17.332,12.123,6.693,3.754,1.564,B2,30 seconds
3,15.930,8.958,2.621,9.364,3.010,3.463,B2,30 seconds
4,13.227,1.261,3.910,0.736,0.421,5.133,B2,30 seconds


In [319]:
# for 5 minutes, read in raw data & process
perr_dfs =[]
source_dir = Path("DaylongSampling/Results/TotalTimeSampled_ProportionbyTotalAudioTime/5MinSamplingInterval/")
files = source_dir.glob('*_TEST_daylong_estimates.txt')
for file in files: 
    print(file)
    df = get_percenterror_df(file, prop_tts = True, feature_arg="word count")
    perr_dfs.append(df)
combined_perr_proptts_random_5min = pd.concat(perr_dfs, ignore_index=True)
combined_perr_proptts_random_5min = combined_perr_proptts_random_5min.reindex(columns = [0.05, 0.1, 0.12, 0.15, 0.18, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, "Transcript"])
combined_perr_proptts_random_5min = clean_for_systematicsampling(combined_perr_proptts_random_5min)
combined_perr_proptts_random_5min["Sampling Interval"] = "5 minutes"
combined_perr_proptts_random_5min.head()

DaylongSampling/Results/TotalTimeSampled_ProportionbyTotalAudioTime/5MinSamplingInterval/B895_010004_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/TotalTimeSampled_ProportionbyTotalAudioTime/5MinSamplingInterval/A787_001107_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/TotalTimeSampled_ProportionbyTotalAudioTime/5MinSamplingInterval/B895_010002_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/TotalTimeSampled_ProportionbyTotalAudioTime/5MinSamplingInterval/BN32_clean.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/TotalTimeSampled_ProportionbyTotalAudioTime/5MinSamplingInterval/A787_001111_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/TotalTimeSampled_ProportionbyTotalAudioTime/5MinSamplingInterval/A787_001109_cleaned.txt_word count_TEST_daylong_estimates.txt


,0.05,0.1,0.15,0.2,0.3,0.4,Transcript,Sampling Interval
0,42.242,21.041,11.252,27.666,10.651,1.383,B2,5 minutes
1,29.135,16.609,22.293,9.667,0.421,8.264,B2,5 minutes
2,21.018,2.870,20.176,2.681,4.026,3.296,B2,5 minutes
3,30.768,44.461,20.020,2.206,6.424,17.262,B2,5 minutes
4,12.341,28.296,27.656,11.326,6.183,13.764,B2,5 minutes


In [321]:
combined_perr_proptts_random = pd.concat([combined_perr_proptts_random_30s, combined_perr_proptts_random_5min],ignore_index=True)
combined_perr_proptts_random.head()

,0.05,0.1,0.15,0.2,0.3,0.4,Transcript,Sampling Interval
0,13.271,6.343,6.323,1.051,5.343,10.084,B2,30 seconds
1,11.734,4.594,6.696,4.911,2.526,0.170,B2,30 seconds
2,14.718,17.332,12.123,6.693,3.754,1.564,B2,30 seconds
3,15.930,8.958,2.621,9.364,3.010,3.463,B2,30 seconds
4,13.227,1.261,3.910,0.736,0.421,5.133,B2,30 seconds


In [ ]:
# write to csv
# combined_perr_proptts_random.to_csv(path_or_buf='DaylongSampling/Results/CSV/Analysis2_1_RandomSamplingComparison_EstimatePercentError.csv', sep = ",", header = True, index = False)

### Estimating: Select Features 

In [324]:
# prep random sampling data 
perr_frequency_density_groups_random = perr_freqdensity_groups.copy()
df_values = perr_frequency_density_groups_random.iloc[:, 0:12]
df_labels = perr_frequency_density_groups_random.iloc[:, 11:]
df_values_clean = clean_for_systematicsampling(df_values)
perr_freqdensity_groups_random_cleaned = pd.concat([df_values_clean.iloc[:, 0:-1], df_labels], axis=1)
perr_freqdensity_groups_random_cleaned.head()

,0.05,0.1,0.15,0.2,0.3,0.4,Transcript,Feature,Frequency Density Group
0,50.0,25.0,66.675,12.5,75.000,37.50,B1,select word,rare
1,100.0,25.0,66.675,37.5,33.325,18.75,B1,select word,rare
2,350.0,75.0,50.000,62.5,25.000,12.50,B1,select word,rare
3,100.0,25.0,16.675,37.5,116.675,43.75,B1,select word,rare
4,250.0,0.0,50.000,37.5,58.325,18.75,B1,select word,rare


In [ ]:
# write to csv
# perr_freqdensity_groups_random_cleaned.to_csv(path_or_buf='DaylongSampling/Results/CSV/Analysis2_2_RandomSamplingComparison_SelectFeatures_EstimatePercentError.csv', sep = ",", header = True, index = False)

## Random Sampling w/out Replacement

### Estimating: Daylong Word Count

In [66]:
### read in 30s data
perr_dfs =[]
source_dir = Path("DaylongSampling/Results/RandomSamplingwithoutReplacement/DaylongWordCounts/30SecondSamplingIntervals")
files = source_dir.glob('*_TEST_daylong_estimates.txt')
for file in files: 
    print(file)
    df = get_percenterror_df(file, prop_tts = True, feature_arg="word count")
    perr_dfs.append(df)
combined_perr_proptts_randomnoreplacement_30sec = pd.concat(perr_dfs, ignore_index=True)
combined_perr_proptts_randomnoreplacement_30sec = combined_perr_proptts_randomnoreplacement_30sec.reindex(columns = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, "Transcript"])
#combined_perr_proptts_random_5min = clean_for_systematicsampling(combined_perr_proptts_random_5min)
combined_perr_proptts_randomnoreplacement_30sec["Sampling Interval"] = "30 seconds"
combined_perr_proptts_randomnoreplacement_30sec.head()

DaylongSampling/Results/RandomSamplingwithoutReplacement/DaylongWordCounts/30SecondSamplingIntervals/B895_010004_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithoutReplacement/DaylongWordCounts/30SecondSamplingIntervals/A787_001107_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithoutReplacement/DaylongWordCounts/30SecondSamplingIntervals/B895_010002_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithoutReplacement/DaylongWordCounts/30SecondSamplingIntervals/BN32_clean.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithoutReplacement/DaylongWordCounts/30SecondSamplingIntervals/A787_001111_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithoutReplacement/DaylongWordCounts/30SecondSamplingIntervals/A787_001109_cleaned.txt_word count_TEST_daylong_estimates.txt


,0.05,0.1,0.15,0.2,0.3,0.4,Transcript,Sampling Interval
0,6.833,1.492,5.685,6.323,2.953,2.955,B2,30 seconds
1,8.979,6.113,4.470,4.573,1.733,5.390,B2,30 seconds
2,0.444,11.522,8.344,0.176,0.053,3.768,B2,30 seconds
3,5.996,1.471,2.526,3.743,0.102,2.186,B2,30 seconds
4,32.492,0.421,2.961,0.104,0.864,0.077,B2,30 seconds


In [67]:
### read in 5 minute data
perr_dfs =[]
source_dir = Path("DaylongSampling/Results/RandomSamplingwithoutReplacement/DaylongWordCounts/5MinuteSamplingIntervals/")
files = source_dir.glob('*_TEST_daylong_estimates.txt')
for file in files: 
    print(file)
    df = get_percenterror_df(file, prop_tts = True, feature_arg="word count")
    perr_dfs.append(df)
combined_perr_proptts_randomnoreplacement_5min = pd.concat(perr_dfs, ignore_index=True)
combined_perr_proptts_randomnoreplacement_5min = combined_perr_proptts_randomnoreplacement_5min.reindex(columns = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, "Transcript"])
#combined_perr_proptts_random_5min = clean_for_systematicsampling(combined_perr_proptts_random_5min)
combined_perr_proptts_randomnoreplacement_5min["Sampling Interval"] = "5 minutes"
combined_perr_proptts_randomnoreplacement_5min.head()

DaylongSampling/Results/RandomSamplingwithoutReplacement/DaylongWordCounts/5MinuteSamplingIntervals/B895_010004_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithoutReplacement/DaylongWordCounts/5MinuteSamplingIntervals/A787_001107_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithoutReplacement/DaylongWordCounts/5MinuteSamplingIntervals/B895_010002_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithoutReplacement/DaylongWordCounts/5MinuteSamplingIntervals/BN32_clean.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithoutReplacement/DaylongWordCounts/5MinuteSamplingIntervals/A787_001111_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithoutReplacement/DaylongWordCounts/5MinuteSamplingIntervals/A787_001109_cleaned.txt_word count_TEST_daylong_estimates.txt


,0.05,0.1,0.15,0.2,0.3,0.4,Transcript,Sampling Interval
0,17.612,5.203,22.091,1.293,6.654,2.328,B2,5 minutes
1,6.602,14.160,10.161,9.096,2.194,2.771,B2,5 minutes
2,17.846,0.071,4.705,5.375,10.280,5.850,B2,5 minutes
3,14.764,9.612,26.492,2.915,4.052,8.159,B2,5 minutes
4,37.437,13.411,8.344,1.762,0.605,1.772,B2,5 minutes


In [68]:
combined_perr_proptts_random_nr = pd.concat([combined_perr_proptts_randomnoreplacement_30sec, combined_perr_proptts_randomnoreplacement_5min],ignore_index=True)
combined_perr_proptts_random_nr.head()

,0.05,0.1,0.15,0.2,0.3,0.4,Transcript,Sampling Interval
0,6.833,1.492,5.685,6.323,2.953,2.955,B2,30 seconds
1,8.979,6.113,4.470,4.573,1.733,5.390,B2,30 seconds
2,0.444,11.522,8.344,0.176,0.053,3.768,B2,30 seconds
3,5.996,1.471,2.526,3.743,0.102,2.186,B2,30 seconds
4,32.492,0.421,2.961,0.104,0.864,0.077,B2,30 seconds


In [ ]:
# write to csv
# combined_perr_proptts_random_nr.to_csv(path_or_buf='DaylongSampling/Results/CSV/Analysis2_1_RandomSamplingwithoutReplacement_EstimatePercentError.csv', sep = ",", header = True, index = False)

### Estimating: Select Features

In [79]:
freqdensity = ["rare", "low", "medium", "high", "very high"]
PROP_TTS = ["0.05","0.1", "0.15","0.2", "0.3", "0.4"] 
source_dir = "DaylongSampling/Results/RandomSamplingwithoutReplacement/Select_Features/"
sourcedirs = [["Rare_Density/Select_Word", "Rare_Density/Utterance_Imitations"], 
              ["Low_Density/Utterance_Exclamations", "Low_Density/Speaker_CHI", "Low_Density/XDS_C"],
              ["Medium_Density/Utterance_Questions", "Medium_Density/Speaker_FC1", "Medium_Density/XDS_T"], 
              ["High_Density/Speaker_FA1_MA1", "High_Density/Speaker_EE1", "High_Density/XDS_A", "High_Density/XDS_T"],
              ["Very_High_Density/Speaker_FA1", "Very_High_Density/XDS_A", "Very_High_Density/XDS_C"]]
select_features = [["select word", "imitations"], 
                   ["exclamations", "target-child vocalizations", "other-child directed speech"],
                   ["questions", "speech from non-target child", "target-child directed speech"], 
                   ["speech from FA1/MA1", "electronic source", "adult directed speech", "target-child directed speech"],
                   ["speech from FA1", "adult directed speech", "other-child directed speech"]]
freqdensity = ["rare", "low", "medium", "high", "very high"]
perr_freqdensity_groups_rs_noreplacement = pd.DataFrame()
for i, fd_group in enumerate(freqdensity):
    fd_group_perr = make_perrdata_featuredensitygrp(source_dir = source_dir,sourcedirs = sourcedirs[i], 
                                                    select_features = select_features[i], density_group = fd_group)
    perr_freqdensity_groups_rs_noreplacement = pd.concat([perr_freqdensity_groups_rs_noreplacement, fd_group_perr],  ignore_index = True)
perr_freqdensity_groups_rs_noreplacement = perr_freqdensity_groups_rs_noreplacement.drop(columns=[0.5, 0.6, 0.7,0.8,0.9])

rare   ['select word', 'imitations']
select word
DaylongSampling/Results/RandomSamplingwithoutReplacement/Select_Features/Rare_Density/Select_Word/B895_010002_cleaned.txt_select_word_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithoutReplacement/Select_Features/Rare_Density/Select_Word/A787_001111_cleaned.txt_select_word_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithoutReplacement/Select_Features/Rare_Density/Select_Word/A787_001109_cleaned.txt_select_word_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithoutReplacement/Select_Features/Rare_Density/Select_Word/A787_001107_cleaned.txt_select_word_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithoutReplacement/Select_Features/Rare_Density/Select_Word/B895_010004_cleaned.txt_select_word_TEST_daylong_estimates.txt
DaylongSampling/Results/RandomSamplingwithoutReplacement/Select_Features/Rare_Density/Select_Word/BN32_clean.txt_select_word_TEST_daylong_estimate

In [ ]:
# write to csv
# perr_freqdensity_groups_rs_noreplacement.to_csv(path_or_buf='DaylongSampling/Results/CSV/Analysis2_2_RandomSamplingwithoutReplacement_SelectFeatures_EstimatePercentError.csv', sep = ",", header = True, index = False)

# Supplemental materials

### removing silences prior to sampling

In [169]:
# read in data 
perr_dfs =[]
source_dir = Path("DaylongSampling/Results/TotalTimeSampled_NoSilences")
files = source_dir.glob('*_TEST_daylong_estimates.txt')
for file in files: 
    print(file)
    df = get_percenterror_df(file, prop_tts = False, feature_arg = "word count")
    perr_dfs.append(df)
combined_perr_nosilences = pd.concat(perr_dfs, ignore_index=True)
combined_perr_nosilences.columns = [30,40,50,60,70,80,90,100,110,120,130,140,150,160,170,180] + ["Transcript"]
combined_perr_nosilences.describe()

DaylongSampling/Results/TotalTimeSampled_NoSilences/B895_010004_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/TotalTimeSampled_NoSilences/A787_001107_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/TotalTimeSampled_NoSilences/B895_010002_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/TotalTimeSampled_NoSilences/BN32_clean.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/TotalTimeSampled_NoSilences/A787_001111_cleaned.txt_word count_TEST_daylong_estimates.txt
DaylongSampling/Results/TotalTimeSampled_NoSilences/A787_001109_cleaned.txt_word count_TEST_daylong_estimates.txt


,30,40,50,60,70,80,90,100,110,120,130,140,150,160,170,180
count,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000
mean,15.206613,11.897625,10.550007,10.388937,9.388632,8.575143,8.187870,7.960963,7.573082,7.506932,7.060177,6.742760,6.445473,6.053995,6.283763,6.054798
std,12.780465,9.407246,8.961015,10.096372,9.000368,7.392846,6.900076,6.752910,6.570373,6.665139,6.258568,6.360298,6.253407,4.954494,5.645234,5.600370
min,0.126000,0.046000,0.195000,0.025000,0.063000,0.015000,0.000000,0.020000,0.009000,0.020000,0.027000,0.033000,0.020000,0.027000,0.003000,0.012000
25%,5.628500,5.163750,4.042000,3.652500,3.294750,3.473000,3.020750,3.063500,2.608250,2.838500,2.598750,2.309250,2.450000,2.228500,2.326500,1.953250
50%,12.572500,9.842500,8.353000,7.600000,7.179000,6.611000,6.640000,6.119000,5.674000,6.050500,5.622000,5.137500,5.018500,4.706500,4.985000,4.531500
75%,21.636000,16.602750,14.940000,14.318500,12.924750,11.985500,11.403250,11.090500,10.921500,10.330750,9.558500,9.287750,8.646500,8.567750,8.513000,7.874750
max,112.381000,83.174000,75.378000,81.473000,87.625000,48.591000,49.019000,45.323000,44.352000,50.817000,41.955000,48.971000,62.348000,29.396000,49.150000,44.507000
